# Assess Fsize Plot From Data Path

只需要改 `DATA_PATHS`。可以给 `~/code/data/<date>/<host>/outputAssessing`，也可以给更上层的 `~/code/data/<date>`，notebook 会递归找 `assess_ta_cdf.csv` 并从目录/`meta.md` 推断 host、expr、timer、fsize、batch。

执行后会在 notebook 里直接显示图，同时把 PNG 和 summary CSV 保存到 `OUTPUT_DIR`。

In [ ]:
from pathlib import Path

# 只改这里：填你手动拉回来的数据目录。可以填一个或多个。
DATA_PATHS = [
    Path.home() / "code/data/20260521",
    # Path.home() / "code/data/20260521/camd9554n2/outputAssessing",
]

OUTPUT_DIR = Path("scripts_plot/outputAssessing/manual_from_path")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import ScalarFormatter
try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

%matplotlib inline

sys.path.insert(0, str(Path("scripts_plot").resolve()))
from assess_plot import collect_expr, format_host_name

TIMER_ORDER = ["tsc", "tsc_asym", "clock_gettime", "mpi_wtime", "cntvct", "cntvct_fence", "cntvcto", "papi", "papix6", "likwid"]
TIMER_COLORS = {
    "tsc": "#1f77b4",
    "tsc_asym": "#17becf",
    "clock_gettime": "#ff7f0e",
    "mpi_wtime": "#2ca02c",
    "cntvct": "#9467bd",
    "cntvct_fence": "#8c564b",
    "cntvcto": "#e377c2",
    "papi": "#d62728",
    "papix6": "#7f7f7f",
    "likwid": "#bcbd22",
}

def ordered(values):
    values = set(values)
    return [x for x in TIMER_ORDER if x in values] + sorted(values - set(TIMER_ORDER))

def infer_host(expr_root: Path) -> str:
    parts = list(expr_root.parts)
    if "outputAssessing" in parts:
        i = parts.index("outputAssessing")
        if i >= 1:
            return parts[i - 1]
    return expr_root.parents[1].name if len(expr_root.parents) > 1 else "unknown"

def find_expr_roots(paths):
    roots = set()
    for raw in paths:
        p = Path(raw).expanduser()
        if not p.exists():
            print(f"WARN missing path: {p}")
            continue
        for cdf in p.rglob("assess_ta_cdf.csv"):
            try:
                roots.add(cdf.parents[4])  # .../<expr>/<batch>/<timer>/<combo>/<walk>/assess_ta_cdf.csv
            except IndexError:
                pass
    return sorted(roots)

def load_from_paths(paths):
    frames = []
    for root in find_expr_roots(paths):
        host = infer_host(root)
        df = collect_expr(root, host)
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["expr_base"] = df["expr"].str.replace(".repeat1", "", regex=False).str.replace(".repeat2", "", regex=False)
    return df


In [ ]:
df = load_from_paths(DATA_PATHS)
if df.empty:
    raise RuntimeError("No assess_ta_cdf.csv found under DATA_PATHS")

# 每个 host/expr/timer/fsize 取最新 batch。这里不按预设参数过滤，避免和你的实测数据冲突。
df = df.sort_values(["host", "expr_base", "timer", "fsize_kib", "batch"])
df_latest = df.groupby(["host", "expr_base", "timer", "fsize_kib"], as_index=False).tail(1).copy()

summary_path = OUTPUT_DIR / "assess_summary_from_path.csv"
df_latest.to_csv(summary_path, index=False)
print(summary_path)
print("rows", len(df_latest))
display(df_latest[["host", "expr", "expr_base", "batch", "timer", "fsize_kib", "interval_ns", "n_measured", "r_l_pct", "r_h_pct", "w1_ns"]].head(20))


In [ ]:
def plot_expr_fsize(sub, expr):
    hosts = list(dict.fromkeys(sub["host"].tolist()))
    fig, axes = plt.subplots(len(hosts), 2, figsize=(13.6, max(3.0, 2.9 * len(hosts))), squeeze=False)
    for row, host in enumerate(hosts):
        hdf = sub[sub["host"] == host]
        for col, (metric, ylabel) in enumerate([("r_l_pct", r"$R_L$ (%)"), ("r_h_pct", r"$R_H$ (%)")]):
            ax = axes[row, col]
            for timer in ordered(hdf["timer"]):
                tdf = hdf[hdf["timer"] == timer].sort_values("fsize_kib")
                ax.plot(tdf["fsize_kib"], tdf[metric], marker="o", linewidth=1.5, markersize=3.8, label=timer, color=TIMER_COLORS.get(timer))
            ax.set_title(format_host_name(host))
            ax.set_xlabel("fsize (KiB)")
            ax.set_ylabel(ylabel)
            if hdf["fsize_kib"].nunique() > 1:
                ax.set_xscale("log", base=2)
                ax.set_xticks(sorted(hdf["fsize_kib"].dropna().unique()))
                ax.get_xaxis().set_major_formatter(ScalarFormatter())
            ax.grid(True, alpha=0.28)
            if not hdf.empty:
                ax.legend(title="timer", fontsize=7, title_fontsize=8, ncols=2)
    fig.suptitle(f"{expr}: $R_L$ and $R_H$ vs fsize", y=0.995)
    fig.tight_layout()
    out = OUTPUT_DIR / f"{expr}_rl_rh_from_path.png"
    fig.savefig(out, dpi=220)
    print(out)
    plt.show()

def plot_expr_wasserstein(sub, expr):
    hosts = list(dict.fromkeys(sub["host"].tolist()))
    fig, axes = plt.subplots(len(hosts), 1, figsize=(7.2, max(3.0, 2.7 * len(hosts))), squeeze=False)
    for ax, host in zip(axes.ravel(), hosts):
        hdf = sub[sub["host"] == host]
        for timer in ordered(hdf["timer"]):
            tdf = hdf[hdf["timer"] == timer].sort_values("fsize_kib")
            ax.plot(tdf["fsize_kib"], tdf["w1_ns"], marker="o", linewidth=1.5, label=timer, color=TIMER_COLORS.get(timer))
        ax.set_title(format_host_name(host))
        ax.set_xlabel("fsize (KiB)")
        ax.set_ylabel("Wasserstein (ns)")
        if hdf["fsize_kib"].nunique() > 1:
            ax.set_xscale("log", base=2)
            ax.set_xticks(sorted(hdf["fsize_kib"].dropna().unique()))
            ax.get_xaxis().set_major_formatter(ScalarFormatter())
        ax.grid(True, alpha=0.28)
        if not hdf.empty:
            ax.legend(title="timer", fontsize=8, title_fontsize=8, ncols=2)
    fig.suptitle(f"{expr}: Wasserstein vs fsize", y=0.995)
    fig.tight_layout()
    out = OUTPUT_DIR / f"{expr}_wasserstein_from_path.png"
    fig.savefig(out, dpi=220)
    print(out)
    plt.show()

for expr in sorted(df_latest["expr_base"].unique()):
    sub = df_latest[(df_latest["expr_base"] == expr) & (df_latest["fsize_kib"] > 0)].copy()
    if sub.empty:
        continue
    plot_expr_fsize(sub, expr)
    plot_expr_wasserstein(sub, expr)
